# Exp 21: Face erase on non-VIP images

Card: [exp21_face_erase.md](../../docs/experiments/exp21_face_erase.md)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

from core.boxes import FACE_BOXES, read_boxes
from core.boxes.pseudo import FACE_SOURCE, FACE_THRESHOLD, pseudo_boxes
from core.training.config import ROOT, TRAIN_DIR
from core.training.config import VIP
from core.training.cv import CVResult
from core.data.dataset import read_train
from core.analysis.gradcam import GradCamFigures
from core.analysis.plots import pr_curves, save_figures, use_style

N_BOXES = 16
N_SAMPLE = 16  # exp19 sample
N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp21_face_erase" / "resnet18.yaml"
EXP16 = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"
EXP20 = FACE_SOURCE
FOLD = 0
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp21_face_erase"

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "exp20": "#2a78d6", "face erase": "#1baf7a"}
use_style()

## Face boxes

In [ ]:
if not FACE_BOXES.exists():  # same as `python -m core.boxes.pseudo --kind=face`
    pseudo_boxes(EXP20, read_train(), FACE_THRESHOLD, classes=("Cat", "Dog")).to_csv(FACE_BOXES, index=False)
boxes = read_boxes(FACE_BOXES)
train = read_train()
train = train[(train["fold"] != FOLD) & ~train[VIP].any(axis=1)]  # erased rows: fold 0 train part, non-VIP
figures = {}
fig, axes = plt.subplots(4, N_BOXES // 2, figsize=(1.1 * N_BOXES, 9), squeeze=False)
for k, cls in enumerate(["Cat", "Dog"]):
    ids = train.loc[train[cls] == 1, "image_id"].sample(N_BOXES, random_state=SEED)
    for ax, image_id in zip(axes[2 * k : 2 * k + 2].flat, ids):
        ax.imshow(Image.open(TRAIN_DIR / f"{image_id}.jpg").convert("RGB").resize((224, 224)))
        for c, (x0, y0, x1, y1) in boxes.get(image_id, []):
            if c == cls:
                ax.add_patch(Rectangle((x0 * 224, y0 * 224), (x1 - x0) * 224, (y1 - y0) * 224, fill=False, color="#eb6834", lw=2))
        ax.set_title(cls, fontsize=8)
for ax in axes.flat:
    ax.set_axis_off()
fig.suptitle(f"Face boxes · Grad-CAM >= {FACE_THRESHOLD} of max, exp20 · fold {FOLD} train part, non-VIP", color="#52514e")
fig.tight_layout()
figures[f"{PREFIX}_face_boxes.png"] = fig
plt.show()

## Training

In [ ]:
results = {"exp16": CVResult.load(EXP16), "exp20": CVResult.load(EXP20)}
results["face erase"] = CVResult.load(CONFIG)

## Grad-CAM Cat, Dog: exp16, exp20, face erase

In [ ]:
cams = {"exp16": GradCamFigures(EXP16), "exp20": GradCamFigures(EXP20), "face erase": GradCamFigures(CONFIG)}
source = "exp16"  # exp19 rows drawn from exp16 OOF
for cls in ["Cat", "Dog"]:
    rows = cams[source].positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen vs trained images

In [ ]:
for cls in VIP:
    for train_view in (False, True):
        fig = cams["face erase"].positives(cls, train=train_view, n=N_POSITIVES)
        split = "train" if train_view else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

## Precision-recall curves

In [ ]:
fig = pr_curves(results, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)